# Breast — SLC transporter factors

Copy of `signature_analysis_breast.ipynb` that views the **SLC transporter** factor
block instead of the cancer neighborhood factors. SLC factors are **cell-autonomous**
(each T cell's own SLC-gene RNA, added like a TF — no spatial diffusion), so the
`custom`-group factors shown in every `view()` ratio plot below are SLC transporters.
The nbhd-only spatial-panel / interaction-heatmap sections are intentionally omitted.

Run `signature_run.py --config breast --factor-group slc` first to populate the
`{name}/fg_slc` cache this notebook reads.

In [ ]:
import sys
import json
from pathlib import Path

_root = next(p for p in Path.cwd().resolve().parents if (p / '.git').exists() or (p / 'setup.py').exists())
for _p in (str(_root), str(_root / 'src')):
    if _p not in sys.path:
        sys.path.insert(0, _p)

from metab_processing.LinearRegression import signature_helpers as sh, signature_batch as sb
from metab_processing.LinearRegression.signature_configs import CONFIGS
from dataclasses import replace

# SLC variant: cell-autonomous SLC transporter factors instead of the cancer
# neighborhood factors. `factor_group='slc'` routes reads to the {name}/fg_slc cache
# (populated by `signature_run.py --config breast --factor-group slc`).
cfg = replace(CONFIGS['breast'], factor_group='slc')

# Which ratio plots `view()` renders (keys of sh.plot_res).
PLOTS = ('ratio', 'variance_ratio', 'meansq_variance_ratio')
# ---- Which cached fit to READ (must match a computed params subdir) ----
# Default = this config's fit (l1 @ penalty 0.005). To read a different fit, override, e.g.:
#   READ = dict(method='ridge', penalty=800)
READ = dict(method=cfg.method, penalty=cfg.penalty, l1_ratio=cfg.l1_ratio)

In [ ]:
membership = json.load(open(Path(cfg.cache_dir) / '_membership.json'))
print(f'{cfg.name}: loaded membership for {len(membership)} gene set(s)')

## SLC factors in this run

`_SLC_FACTORS` lists the SLC genes added as cell-autonomous predictors. A target
signature gene that is itself an SLC factor is dropped from that target (shown as
`slc_removed` above) to avoid a self-term in both X and y.

In [ ]:
slc = membership.get('_SLC_FACTORS', {})
print(f"SLC factors ({slc.get('total', 0)}): {slc.get('present', [])}")

## breast: target signatures -- membership, missing, present, factor-removed, remaining

In [ ]:
print(f'===== {cfg.name} target signatures: membership + factor-removal =====')
for name in list(cfg.target_signatures) + list(cfg.neg_controls) + list(cfg.manual_sets):
    m = membership.get(name, {})
    print(f'\n{name}')
    print(f"  present ({len(m.get('present', []))}): {m.get('present', [])}")
    print(f"  missing ({len(m.get('missing', []))}): {m.get('missing', [])}")
    print(f"  total: {m.get('total', 0)}")
    print(f"  factor_removed ({len(m.get('factor_removed', []))}): {m.get('factor_removed', [])}")
    print(f"  remaining ({len(m.get('remaining', []))}): {m.get('remaining', [])}")
    print(f"  slc_removed ({len(m.get('slc_removed', []))}): {m.get('slc_removed', [])}")

## View helper -- `view(sig)` renders every cached run-spec for one signature

In [ ]:
TARGETS = sb.all_targets(cfg)   # targets + neg controls + manual sets (+ colon single gene)
specs = sb.runspecs(cfg)


def view(sig):
    for spec in specs:
        res = sb.load_res(cfg, sig, spec['label'], **READ)
        if res is None:
            print('not yet computed:', sig, spec['label'])
            continue
        sh.plot_res(res, highlight_genes=membership.get(sig, {}).get('present', []),
                    title=f"{sig} | {spec['label']}", plots=PLOTS)

## Metabolic targets (predicted like the T-cell signatures)

In [ ]:
view('HALLMARK_HYPOXIA')

In [ ]:
view('HALLMARK_GLYCOLYSIS')

In [ ]:
view('HALLMARK_OXIDATIVE_PHOSPHORYLATION')

In [ ]:
view('HALLMARK_FATTY_ACID_METABOLISM')

In [ ]:
view('HALLMARK_MTORC1_SIGNALING')

## Subsample stats from cached raw betas -- call `stats('SIGNATURE_NAME')`

In [ ]:
# 95% CI / selection-frequency / stability, computed directly from the cached raw subsample
# betas. (CI at n_sub resamples is coarse; under L1 betas pile at 0 so selection_freq + a
# CI excluding 0 are the honest signals, p_stability is a conservative heuristic.)
from IPython.display import display

def stats(sig):
    for spec in specs:
        betas = sb.load_betas(cfg, sig, spec['label'], **READ)
        if betas is None:
            print('not yet computed:', sig, spec['label']); continue
        print(f"\n=== {sig} | {spec['label']} ===")
        display(sh.subsample_stats(betas).sort_values('selection_freq', ascending=False).head(25))

## T-cell TARGET signatures

### HALLMARK_IL2_STAT5_SIGNALING

In [ ]:
view('HALLMARK_IL2_STAT5_SIGNALING')

### KEGG_T_CELL_RECEPTOR_SIGNALING_PATHWAY

In [ ]:
view('KEGG_T_CELL_RECEPTOR_SIGNALING_PATHWAY')

### GOBP_T_CELL_MEDIATED_CYTOTOXICITY

In [ ]:
view('GOBP_T_CELL_MEDIATED_CYTOTOXICITY')

### KEGG_NATURAL_KILLER_CELL_MEDIATED_CYTOTOXICITY

In [ ]:
view('KEGG_NATURAL_KILLER_CELL_MEDIATED_CYTOTOXICITY')

### HALLMARK_TNFA_SIGNALING_VIA_NFKB

In [ ]:
view('HALLMARK_TNFA_SIGNALING_VIA_NFKB')

### HALLMARK_INTERFERON_GAMMA_RESPONSE

In [ ]:
view('HALLMARK_INTERFERON_GAMMA_RESPONSE')

### HALLMARK_INTERFERON_ALPHA_RESPONSE

In [ ]:
view('HALLMARK_INTERFERON_ALPHA_RESPONSE')

### GOBP_CELL_KILLING

In [ ]:
view('GOBP_CELL_KILLING')

### GOBP_GRANZYME_MEDIATED_PROGRAMMED_CELL_DEATH_SIGNALING_PATHWAY

In [ ]:
view('GOBP_GRANZYME_MEDIATED_PROGRAMMED_CELL_DEATH_SIGNALING_PATHWAY')

### GOBP_LYMPHOCYTE_MEDIATED_IMMUNITY

In [ ]:
view('GOBP_LYMPHOCYTE_MEDIATED_IMMUNITY')

## Negative controls

### HALLMARK_PANCREAS_BETA_CELLS

In [ ]:
view('HALLMARK_PANCREAS_BETA_CELLS')

### HALLMARK_SPERMATOGENESIS

In [ ]:
view('HALLMARK_SPERMATOGENESIS')

## MANUAL_ISLET_NEG_CONTROL (manual negative control)

In [ ]:
view('MANUAL_ISLET_NEG_CONTROL')

## Spatial SLC-expression panels

SLC factors are **cell-autonomous** (each T cell's own SLC-gene RNA), so there is no
other-cell-type neighborhood to map. Instead, spatially plot the T cells coloured by
their target gene-set score (others greyed), then one panel per **significant SLC
factor** (subsample interval excludes 0): the T cells coloured by that SLC gene's own
`normalized_count` expression -- the exact per-cell value the regression saw.

The SLC factors are re-added to `adata` here (`sh.add_slc_factors`) so `obsm['x_custom']`
carries them for the residual / heatmap sections below, mirroring how the nbhd notebook
re-adds its cancer factors.

In [ ]:
import numpy as np
import scanpy as sc
import matplotlib.pyplot as plt
from metab_processing.LinearRegression.build_x import LAYER, ensure_lognorm_layer

adata = sc.read_h5ad(cfg.adata_path)
ensure_lognorm_layer(adata)
var = set(adata.var_names)
CELL_TYPE_COL = cfg.cell_type_col
TCELL_TYPES = list(cfg.t_cell_types)

# Re-add the cell-autonomous SLC factors -> obsm['x_custom'] + uns['x_custom_cols'].
# (Same genes the cached fit used; `slc_genes` also drives the panel / heatmap factor lists.)
slc_genes = sh.add_slc_factors(adata, cfg.slc_pattern)
slc_set = set(slc_genes)

# Raw gene sets behind each target, with SLC-pattern genes dropped to mirror the cached
# pipeline's `slc_removed` (so the refit y below never contains a gene that is also an SLC
# factor -- no self-term). Used by the residual fits and the spatial target-score panel.
library = sh.load_msigdb(cfg.dbver, cfg.categories)
sets = {n: [g for g in library.get(n, []) if g in var]
        for n in list(cfg.target_signatures) + list(cfg.neg_controls)}
sets.update({n: [g for g in gl if g in var] for n, gl in cfg.manual_sets.items()})
sets = {n: [g for g in gl if g not in slc_set] for n, gl in sets.items()}
print(adata.n_obs, 'cells |', len(slc_genes), 'SLC factors:', slc_genes)

In [ ]:
def gene_set_score(adata, genes):
    """Per-cell sum of a gene set's `normalized_count` expression (present genes only). For a
    single SLC gene this is exactly that gene's SLC factor value in obsm['x_custom']."""
    present = [g for g in genes if g in adata.var_names]
    if not present:
        return np.zeros(adata.n_obs)
    return adata[:, present].to_df(LAYER)[present].sum(axis=1).to_numpy()


def significant_slc_factors(res, slc_genes, touch_zero=True):
    """From a target's `res`, the SLC factors whose subsample interval excludes 0, ordered by
    |beta| descending. Returns a list of SLC gene names."""
    if res is None or res.empty:
        return []
    sig = sh.filter_nonzero_interval(res, touch_zero=touch_zero).copy()
    sig = sig.reindex(sig['beta'].abs().sort_values(ascending=False).index)
    sset = set(slc_genes)
    return [f for f in sig['factor'] if f in sset]


def tcell_scored_genes(sig):
    """Genes actually scored for a target's pseudo-gene (the y constituents): the SLC-stripped
    gene list split EXACTLY as add_gene_signature does (`signature_factor_split`), so panel 0
    equals the fitted y. Falls back to membership `remaining` (minus SLC) for any target not
    in `sets` (e.g. a single-gene colon target)."""
    if sig in sets:
        return list(sh.signature_factor_split(adata, sets[sig])[1])
    m = membership.get(sig, {})
    genes = m.get('remaining') or m.get('present', [])
    return [g for g in genes if g not in slc_set]


def plot_spatial_slc_panels(adata, cell_type_col, tcell_types, tcell_genes, slc_factors,
                            point_size=3, cmap='viridis', tcell_label='T cells'):
    """Row of spatial panels over the T cells: panel 0 = T cells coloured by `tcell_genes`
    score; then one panel per SLC gene in `slc_factors`, T cells coloured by that gene's own
    expression. Non-T cells greyed. Display capped at p95. Returns the fig."""
    xy = np.asarray(adata.obsm['spatial'])
    ct = adata.obs[cell_type_col].astype(str).to_numpy()
    tmask = np.isin(ct, [str(t) for t in tcell_types])

    def _panel(ax, values, title):
        ax.scatter(xy[~tmask, 0], xy[~tmask, 1], s=point_size * 0.5, c='lightgrey', linewidths=0)
        vmax = None
        if tmask.sum():                       # SLC expression is sparse -> p95 is often 0;
            p95 = float(np.nanpercentile(values[tmask], 95))   # fall back to the max so the
            vmax = p95 if p95 > 0 else (float(values[tmask].max()) or None)  # expressing cells show
        sc_ = ax.scatter(xy[tmask, 0], xy[tmask, 1], s=point_size, c=values[tmask], cmap=cmap,
                         vmax=vmax, linewidths=0)
        plt.colorbar(sc_, ax=ax, shrink=0.7)
        ax.set_title(title, fontsize=9); ax.set_aspect('equal')
        ax.set_xticks([]); ax.set_yticks([])

    n = 1 + len(slc_factors)
    fig, axes = plt.subplots(1, n, figsize=(5 * n, 5), squeeze=False)
    axes = axes[0]
    _panel(axes[0], gene_set_score(adata, tcell_genes), f'{tcell_label}: target gene-set score')
    for ax, g in zip(axes[1:], slc_factors):
        _panel(ax, gene_set_score(adata, [g]), f'{tcell_label}\n{g}')
    fig.tight_layout()
    return fig

## Example -- pick a target signature, show its significant SLC factors

In [ ]:
SIG = 'HALLMARK_GLYCOLYSIS'                      # <- any target in TARGETS
LABEL = sb.runspecs(cfg)[0]['label']            # breast has one T type

res = sb.load_res(cfg, SIG, LABEL, **READ)
factors = significant_slc_factors(res, slc_genes)
print(f'{SIG} | {LABEL}: {len(factors)} significant SLC factors: {factors}')

if res is not None and factors:
    plot_spatial_slc_panels(adata, CELL_TYPE_COL, TCELL_TYPES,
                            tcell_scored_genes(SIG), factors[:8]);   # cap for a readable row

## Spatial panels for every target

In [ ]:
def spatial_view(sig, top=8):
    label = sb.runspecs(cfg)[0]['label']
    res = sb.load_res(cfg, sig, label, **READ)
    if res is None:
        print('not yet computed:', sig); return
    factors = significant_slc_factors(res, slc_genes)
    if not factors:
        return
    print(f'{sig}: {len(factors)} significant SLC factors')
    fig = plot_spatial_slc_panels(adata, CELL_TYPE_COL, TCELL_TYPES,
                                  tcell_scored_genes(sig), factors[:top])
    fig.suptitle(sig, y=1.02)


for sig in TARGETS:
    spatial_view(sig)

## Target vs SLC-factor & residual plots

Same three-panel decomposition as the nbhd notebook, but the `custom` factor group is now
the SLC transporters (both live in `obsm['x_custom']`, so the machinery is identical):
  **PLOT 1** target score vs an SLC factor (binned, with R² / Spearman);
  **PLOT 2** residual using the CACHED full-model's GRN (non-SLC) betas -- no training;
  **PLOT 3** residual of a fresh GRN-only refit (`custom=None`).
Plots 2 & 3 vs an SLC factor show what that transporter explains *beyond* the GRN model.

In [ ]:
# ---- Residual plots: what a model WITHOUT the SLC factors leaves on the table ----
# Decomposed so the EXPENSIVE fit (per signature) is separate from the CHEAP binned plotting
# (per factor + n_bins): train once -> re-plot any SLC factor / any n_bins, no refit.
from dataclasses import dataclass
from metab_processing.LinearRegression.build_x import get_gene_factors, LAYER
from sklearn.linear_model import Lasso, Ridge, ElasticNet, LinearRegression
from scipy.stats import spearmanr
import time as _time


def _binned_mean_sem(x, y, n_bins=10):
    """Equal-count (quantile) bins of x -> per-bin mean +/- SEM of y. Returns (cx, my, se)."""
    x = np.asarray(x, float); y = np.asarray(y, float)
    edges = np.unique(np.quantile(x, np.linspace(0, 1, n_bins + 1)))
    b = np.clip(np.digitize(x, edges[1:-1]), 0, len(edges) - 2)
    cx, my, se = [], [], []
    for k in range(len(edges) - 1):
        yk, xk = y[b == k], x[b == k]
        if yk.size:
            cx.append(xk.mean()); my.append(yk.mean())
            se.append(yk.std(ddof=1) / np.sqrt(yk.size) if yk.size > 1 else 0.0)
    return np.array(cx), np.array(my), np.array(se)


def _pearson_r2(x, y):
    x = np.asarray(x, float); y = np.asarray(y, float)
    if x.size < 2 or x.std() == 0 or y.std() == 0:
        return np.nan
    r = np.corrcoef(x, y)[0, 1]
    return float(r * r)


def _spearman(x, y):
    x = np.asarray(x, float); y = np.asarray(y, float)
    if x.size < 2 or np.unique(x).size < 2 or np.unique(y).size < 2:
        return np.nan
    return float(spearmanr(x, y).correlation)


def binned_scatter(x, y, ax=None, *, n_bins=10, xlabel='', ylabel='', title='', r2=None,
                   r2_label='$R^2$', rho=None, scatter_max=20000):
    """CHEAP plotting primitive -- raw scatter + binned mean +/- SEM. `n_bins` lives ONLY here,
    so re-binning never needs a refit. `plt.style.context('default')` isolates these plots from
    the global seaborn theme that the ratio plots (`sns.set_theme`) leave set."""
    with plt.style.context('default'):
        if ax is None:
            _, ax = plt.subplots(figsize=(7, 5))
        xs, ys = np.asarray(x), np.asarray(y)
        if scatter_max and xs.size > scatter_max:   # subsample only the scatter (speed); bins use all
            idx = np.random.default_rng(0).choice(xs.size, scatter_max, replace=False)
            xs, ys = xs[idx], ys[idx]
        ax.scatter(xs, ys, s=6, alpha=0.2, color='grey', label=f'cells (n={len(x)})')
        cx, my, se = _binned_mean_sem(x, y, n_bins=n_bins)
        ax.errorbar(cx, my, yerr=se, fmt='o-', color='#D55E00', capsize=3, lw=2,
                    label=f'binned mean +/- SEM ({len(cx)} bins)')
        ax.set_xlabel(xlabel); ax.set_ylabel(ylabel); ax.set_title(title)
        _ann = []
        if r2 is not None and np.isfinite(r2):
            _ann.append(f'{r2_label} = {r2:.3g}')
        if rho is not None and np.isfinite(rho):
            _ann.append(f'Spearman $\\rho$ = {rho:.3g}')
        if _ann:
            ax.text(0.03, 0.97, '\n'.join(_ann), transform=ax.transAxes, va='top', ha='left',
                    fontsize=10, bbox=dict(boxstyle='round', fc='white', ec='0.7', alpha=0.85))
        ax.legend(); plt.tight_layout(); plt.show()
    return ax


@dataclass
class SignatureFit:
    """Saveable per-cell fit for one signature (holds NO binning). Reuse across factors/n_bins."""
    target: str; cell_types: list; cell_mask: object; obs_names: object; factors: list
    y: object; yhat: object; resid: object; beta: object; intercept: float
    mu: object; sd: object; r2: float; settings: dict


def _make_model(method, penalty, l1_ratio):
    if method == 'l1':      return Lasso(alpha=penalty, fit_intercept=True)
    if method == 'ridge':   return Ridge(alpha=penalty, fit_intercept=True)
    if method == 'elastic': return ElasticNet(alpha=penalty, l1_ratio=l1_ratio, fit_intercept=True)
    if method == 'OLS':     return LinearRegression(fit_intercept=True)
    raise ValueError(f'unknown method {method!r}')


def fit_signature_residuals(target, *, cell_types=None, method='l1', penalty=0.005,
                            l1_ratio=0.5, standardize=True, metabolites=(), custom=None,
                            keep_when_overlap='factor', verbose=True):
    """EXPENSIVE step -- fit the `target` pseudo-gene on its factor matrix over the T cells and
    return a saveable SignatureFit with per-cell y / yhat / resid (no binning). `custom=None`
    -> NO SLC factors (the 'what's captured without them' model); `custom='all'` ->
    include them. Mirrors least_squares._fit's D=None branch (z-score X over the selected cells
    when `standardize`, then sklearn with fit_intercept=True) so predictions -- hence residuals
    -- are exposed (the pipeline's fit returns betas only). Residuals are per-cell and
    factor-independent, so one fit re-plots against ANY SLC factor with no refit.
    NOTE: y is the pseudo-gene score (keep_when_overlap='factor' drops signature genes that
    are themselves factors), so plot 1 here differs from a raw sum of the signature genes."""
    def _log(msg, t):
        if verbose:
            print(f'    [fit {target[:30]}] {msg} ({_time.time() - t:.1f}s)', flush=True)
    cell_types = list(cfg.t_cell_types) if cell_types is None else list(cell_types)
    cell_mask = adata.obs[cfg.cell_type_col].astype(str).isin([str(c) for c in cell_types]).to_numpy()
    if cell_mask.sum() == 0:
        raise ValueError(f'no cells for cell_types={cell_types} in obs[{cfg.cell_type_col!r}]')
    if verbose:
        print(f'fit_signature_residuals({target!r}, custom={custom}): {int(cell_mask.sum())} cells',
              flush=True)
    t0 = _time.time(); t = t0
    # Fit on the T-cell SUBSET only -> add_gene_signature copies far fewer rows (it hstacks the
    # score onto X + every layer, which is the slow part on the full 170k-cell adata).
    ad2 = sh.add_gene_signature(adata[cell_mask], target, positive=sets[target],
                                keep_when_overlap=keep_when_overlap, verbose=False)
    _log('add_gene_signature (pseudo-gene on T-cell subset)', t); t = _time.time()
    X = get_gene_factors(ad2, target, metabs=list(metabolites) or None, custom=custom)
    _log(f'get_gene_factors ({X.shape[0]} cells x {X.shape[1]} factors, custom={custom})', t); t = _time.time()
    Xv = X.to_numpy(float)
    y = ad2[:, target].to_df(LAYER)[target].to_numpy().astype(float)
    _log('densify X / score y', t); t = _time.time()
    mu = Xv.mean(0); sd = np.where(Xv.std(0) == 0, 1.0, Xv.std(0))
    Xz = (Xv - mu) / sd if standardize else Xv
    model = _make_model(method, penalty, l1_ratio).fit(Xz, y)
    yhat = model.predict(Xz)
    _log(f'{method} fit', t)
    if verbose:
        print(f'  fit done in {_time.time() - t0:.1f}s', flush=True)
    return SignatureFit(target=target, cell_types=cell_types, cell_mask=cell_mask,
                        obs_names=np.asarray(ad2.obs_names), factors=list(X.columns),
                        y=y, yhat=yhat, resid=y - yhat, beta=np.asarray(model.coef_, float),
                        intercept=float(model.intercept_), mu=mu, sd=sd,
                        r2=float(model.score(Xz, y)),
                        settings=dict(source='refit', method=method, penalty=penalty, l1_ratio=l1_ratio,
                                      standardize=standardize, metabolites=list(metabolites),
                                      custom=custom, keep_when_overlap=keep_when_overlap))


def grn_residual(target, *, cell_types=None, keep_when_overlap='factor', standardize=None,
                 label=None, verbose=True, **read):
    """PLOT-2 model -- NO TRAINING. Residual of the signature using the CACHED full-model's
    NON-SLC (GRN) betas only. Loads the cached joint fit (`sb.load_res`; the pipeline
    fit uses metabolites=[] + custom='all'), keeps only its GRN (non-`x_custom`) coefficients,
    applies them to the standardized GRN factor matrix, and takes resid = y - GRN prediction.
    The SLC factors' contribution is NOT subtracted, so the residual still carries
    whatever they explain -- plot it vs a SLC factor to see that. These GRN betas are
    AS SHRUNK by the SLC factors in the joint fit, unlike fit_signature_residuals(custom=None)
    (plot 3) which REFITS GRN alone. The cached fit's intercept isn't stored, so the best
    constant is used (only shifts the residual; the vs-factor trend / R^2 / Spearman are
    unaffected). Returns a SignatureFit (settings['source']='cached_grn_betas').
    CAVEATS: use the DEFAULT cell_types (the cache's fit cells) -- a different population mixes
    cached betas with a foreign standardization. The reported model R^2 uses the L1-shrunk JOINT
    GRN betas (not an independent refit), so it can be low/negative; don't compare it directly to
    plot 3's refit R^2. (x and resid both index the GLOBAL adata at plot time -- don't reload or
    reorder adata between fit and plot.)"""
    read = {**READ, **read}
    standardize = cfg.standardize if standardize is None else standardize   # match the cached fit
    label = sb.runspecs(cfg)[0]['label'] if label is None else label
    res = sb.load_res(cfg, target, label, **read)
    if res is None:
        raise ValueError(f'grn_residual: no cached res for {target!r} | {label} | {read}. Plot 2 '
                         'reuses the cached full-model betas -- compute that fit, or fit here with '
                         'fit_signature_residuals(custom="all").')
    beta_by_factor = dict(zip(res['factor'], res['beta']))
    cell_types = list(cfg.t_cell_types) if cell_types is None else list(cell_types)
    cell_mask = adata.obs[cfg.cell_type_col].astype(str).isin([str(c) for c in cell_types]).to_numpy()
    if cell_mask.sum() == 0:
        raise ValueError(f'no cells for cell_types={cell_types} in obs[{cfg.cell_type_col!r}]')
    ad2 = sh.add_gene_signature(adata[cell_mask], target, positive=sets[target],
                                keep_when_overlap=keep_when_overlap, verbose=False)
    y = ad2[:, target].to_df(LAYER)[target].to_numpy().astype(float)
    Xg = get_gene_factors(ad2, target, metabs=None, custom=None)      # GRN (non-SLC) factors only
    factors = list(Xg.columns)
    Xv = Xg.to_numpy(float)
    mu = Xv.mean(0); sd = np.where(Xv.std(0) == 0, 1.0, Xv.std(0))
    Xz = (Xv - mu) / sd if standardize else Xv
    beta = np.array([beta_by_factor.get(f, 0.0) for f in factors], float)
    missing = [f for f in factors if f not in beta_by_factor]
    if missing and verbose:
        print(f'  [grn_residual] {len(missing)}/{len(factors)} GRN factor(s) absent from cached '
              f'res (beta=0): {missing[:5]}', flush=True)
    yhat = Xz @ beta
    intercept = float(np.mean(y - yhat))             # cached fit's intercept not stored -> best constant
    resid = y - (yhat + intercept)
    ss_res = float(np.sum(resid ** 2)); ss_tot = float(np.sum((y - y.mean()) ** 2))
    r2 = (1.0 - ss_res / ss_tot) if ss_tot > 0 else np.nan
    return SignatureFit(target=target, cell_types=cell_types, cell_mask=cell_mask,
                        obs_names=np.asarray(ad2.obs_names), factors=factors,
                        y=y, yhat=yhat + intercept, resid=resid, beta=beta, intercept=intercept,
                        mu=mu, sd=sd, r2=r2,
                        settings=dict(source='cached_grn_betas', label=label,
                                      standardize=standardize, custom=None,
                                      keep_when_overlap=keep_when_overlap))


def _custom_factor_values(factor, cell_mask):
    cols = list(adata.uns['x_custom_cols'])
    if factor not in cols:
        raise KeyError(f"{factor!r} not in x_custom_cols; e.g. {[c for c in cols if 'SLC' in c][:5]}")
    return np.asarray(adata.obsm['x_custom'])[:, cols.index(factor)].astype(float)[cell_mask]


def plot_target_vs_factor(fit, factor, *, n_bins=10, exclude_zero=False, ax=None):
    """PLOT 1: target score (fit.y) vs a SLC factor, binned, with R^2 and Spearman rho.
    `exclude_zero` drops cells where the FACTOR is exactly 0 (the zero mass that collapses the
    equal-count bins / smears into the first bin)."""
    x = _custom_factor_values(factor, fit.cell_mask); y = fit.y
    if exclude_zero:
        keep = x != 0; x, y = x[keep], y[keep]
    return binned_scatter(x, y, ax=ax, n_bins=n_bins, xlabel=factor,
                          ylabel=f'{fit.target} score', title=f'{fit.target}  vs  {factor}',
                          r2=_pearson_r2(x, y), r2_label='$R^2$(factor, score)', rho=_spearman(x, y))


def plot_residual_vs_factor(fit, factor, *, n_bins=10, exclude_zero=False, ax=None):
    """PLOTS 2 & 3: residual of the fitted model (fit.resid) vs a SLC factor, binned.
    With a `custom=None` fit this is 'what's left after the gene-only model'. R^2 = r^2(factor,
    resid) = variance this SLC factor explains BEYOND the model's factors. Reuse one
    fit across many factors / n_bins (no refit). `exclude_zero` drops cells where the FACTOR is
    0 (fixes the zero-inflation bin collapse; the residuals themselves are continuous)."""
    x = _custom_factor_values(factor, fit.cell_mask); y = fit.resid
    if exclude_zero:
        keep = x != 0; x, y = x[keep], y[keep]
    _src = fit.settings.get('source')
    nb = ('cached GRN betas' if _src == 'cached_grn_betas'
          else 'no-SLC refit' if fit.settings.get('custom') is None else 'full')
    return binned_scatter(x, y, ax=ax, n_bins=n_bins, xlabel=factor,
                          ylabel=f'{fit.target} residual',
                          title=f'{fit.target} residual  ({nb} model R²={fit.r2:.3g})  vs  {factor}',
                          r2=_pearson_r2(x, y), r2_label='$R^2$(factor, resid)', rho=_spearman(x, y))


def plot_signature_vs_factor(target, factor, *, n_bins=10, exclude_zero=False,
                             grn_fit=None, none_fit=None, show_plot3=True, cell_types=None,
                             standardize=None, keep_when_overlap='factor',
                             method='l1', penalty=0.005, l1_ratio=0.5):
    """Three panels for (target, factor), all binned over the T cells vs `factor`:
      PLOT 1  target score vs factor (+ R^2, Spearman).
      PLOT 2  residual using the CACHED full-model GRN (non-SLC) betas -- NO training
              (`grn_residual`): what the SLC factors explain on top of the GRN model.
      PLOT 3  residual of a FRESH no-SLC refit (`fit_signature_residuals(custom=None)`).
    Pass `grn_fit` / `none_fit` to reuse saved fits (re-plot other factors / n_bins, no refit);
    `show_plot3=False` skips the (training) plot 3. Returns (grn_fit, none_fit)."""
    standardize = cfg.standardize if standardize is None else standardize
    if grn_fit is None:
        print('plots 1&2: cached GRN betas (no training) ...', flush=True)
        grn_fit = grn_residual(target, cell_types=cell_types, standardize=standardize,
                               keep_when_overlap=keep_when_overlap)
    print('plot 1/3: target score vs factor ...', flush=True)
    plot_target_vs_factor(grn_fit, factor, n_bins=n_bins, exclude_zero=exclude_zero)
    print('plot 2/3: residual (cached GRN betas) vs factor ...', flush=True)
    plot_residual_vs_factor(grn_fit, factor, n_bins=n_bins, exclude_zero=exclude_zero)
    if show_plot3:
        if none_fit is None:
            print('plot 3/3: training no-SLC model ...', flush=True)
            none_fit = fit_signature_residuals(target, custom=None, cell_types=cell_types,
                                               method=method, penalty=penalty, l1_ratio=l1_ratio,
                                               standardize=standardize,
                                               keep_when_overlap=keep_when_overlap)
        else:
            print('plot 3/3: residual (no-SLC refit, reused) vs factor ...', flush=True)
        plot_residual_vs_factor(none_fit, factor, n_bins=n_bins, exclude_zero=exclude_zero)
    return grn_fit, none_fit


In [ ]:
# 3 panels: (1) target score vs SLC factor; (2) residual from the CACHED full-model GRN betas
# (no training); (3) residual of a fresh GRN-only refit. Returns both reusable fits.
TARGET = 'HALLMARK_GLYCOLYSIS'
LABEL = sb.runspecs(cfg)[0]['label']
_res = sb.load_res(cfg, TARGET, LABEL, **READ)
if _res is None:
    print('not yet computed:', TARGET,
          '-- run signature_run.py --config breast --factor-group slc')
else:
    # default to the target's TOP significant SLC factor (falls back to the first SLC gene)
    _sig = significant_slc_factors(_res, slc_genes)
    FACTOR = _sig[0] if _sig else (slc_genes[0] if slc_genes else 'SLC2A1')
    print('factor:', FACTOR)
    grn_fit, none_fit = plot_signature_vs_factor(TARGET, FACTOR)

# Reuse the saved fits for a DIFFERENT SLC factor / n_bins -- no recompute:
# if len(slc_genes) > 1:
#     plot_residual_vs_factor(grn_fit,  slc_genes[1], n_bins=15)   # plot-2 style
#     plot_residual_vs_factor(none_fit, slc_genes[1], n_bins=15)   # plot-3 style

## Interaction heatmap — (target signature × SLC factor) pairings

Each cell = ONE **(target signature, SLC factor) pairing**: the regression coefficient
(`beta`, full fit) of that SLC transporter factor when predicting the target. **Load**
builds a tidy one-row-per-pairing table from the cache; **plot** draws the beta grid
(y = target, x = SLC factor).

**Stars = how robustly the pairing's coefficient stays one-signed across the subsample
refits** (`min_trim` = smallest number of extreme subsamples dropped from EACH end before
the remaining range excludes 0): ★★★ `min_trim` 0; ★★ drop 1/side; ★ drop 2/side.
Significant = ≥ `min_stars` (default 1). `inspect_pair(target, factor)` shows the trimmed
bounds behind any cell.

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt

RUN_LABEL = f'{cfg.t_cell_types[0]}_only'   # breast: single T type
HEAT_READ = dict(READ)                       # which cached fit to read (see READ, top of nb)
Y_TARGETS = list(cfg.target_signatures) + list(cfg.neg_controls) + list(cfg.manual_sets)


def _min_trim_to_exclude(b):
    """Smallest number of extreme points to drop from EACH end of a factor's subsample betas
    before the remaining range excludes 0 (strictly one-signed). 0 = the full set never
    crosses/touches 0; np.inf = still crosses even trimmed to the center. (Monotonic: once an
    interval is one-signed, trimming further keeps it one-signed.) NaN subsamples dropped."""
    b = np.asarray(b, dtype=float); b = b[~np.isnan(b)]
    n = b.size
    if n == 0:
        return np.nan
    s = np.sort(b)
    for t in range((n - 1) // 2 + 1):
        if s[t] > 0 or s[-(t + 1)] < 0:
            return float(t)
    return np.inf


def load_interactions(targets=None, label=RUN_LABEL, **read):
    """Tidy one-row-per-(target, factor) table for ONE run `label`, from the cache. Columns:
    target, factor, beta (full-fit coefficient = plotted value), the subsample CI summary
    from `res`, and `min_trim` (from the raw per-subsample beta sidecar `load_betas`). If the
    sidecar is absent, `min_trim` falls back to 0 when res's full range [min,max] excludes 0
    else NaN -- so only the 3-star 'no crossing' level is distinguishable then."""
    read = {**HEAT_READ, **read}
    targets = Y_TARGETS if targets is None else targets
    out = []
    for t in targets:
        res = sb.load_res(cfg, t, label, **read)
        if res is None:
            print('skip (no cached res):', t, '|', label); continue
        r = res.copy(); r.insert(0, 'target', t); r['run'] = label
        betas = sb.load_betas(cfg, t, label, **read)
        if betas is not None:
            mt = {f: _min_trim_to_exclude(betas[f].to_numpy()) for f in betas.columns}
            r['min_trim'] = r['factor'].map(mt)
        elif 'min_subsample_beta' in r.columns:
            lo, hi = r['min_subsample_beta'], r['max_subsample_beta']
            r['min_trim'] = np.where((lo > 0) | (hi < 0), 0.0, np.nan)
        else:
            r['min_trim'] = np.nan
        out.append(r)
    return pd.concat(out, ignore_index=True) if out else pd.DataFrame()


inter = load_interactions()
print('rows:', len(inter), '| targets:', inter['target'].nunique() if len(inter) else 0,
      '| distinct factors:', inter['factor'].nunique() if len(inter) else 0)
if len(inter):
    print('columns:', list(inter.columns))


In [ ]:
# SLC transporter factors for the x-axis (the cell-autonomous custom-group columns).
SLC_FACTORS = set(slc_genes)

def is_slc_factor(factor):
    return str(factor) in SLC_FACTORS


def default_stars(row):
    """Stars from `min_trim` (smallest #subsamples dropped from each end before the interval
    excludes 0):
       3 = min_trim 0   -> no subsample crosses/touches 0 (full range one-signed)
       2 = min_trim <= 1 -> excludes 0 after dropping the top-1 and bottom-1 subsample
       1 = min_trim <= 2 -> excludes 0 after dropping the top-2 and bottom-2
       else 0.  Change the cutoffs with your own `stars=`(row -> int), e.g.
       stars=lambda r: 3 if r['min_trim']==0 else 2 if r['min_trim']<=3 else 1 if r['min_trim']<=5 else 0"""
    m = row.get('min_trim')
    if pd.isna(m):
        return 0
    if m <= 0:
        return 3
    if m <= 1:
        return 2
    if m <= 2:
        return 1
    return 0


def inspect_pair(target, factor, label=RUN_LABEL, **read):
    """Everything behind ONE (target, factor) cell: point beta, and -- from the raw subsample
    sidecar -- the sign counts, `min_trim`, and the trimmed bounds at drop 0/1/2 per side.
    Falls back to the res interval when the sidecar is absent. Returns the res row."""
    read = {**HEAT_READ, **read}
    res = sb.load_res(cfg, target, label, **read)
    if res is None:
        print('no cached res for', target, '|', label); return
    sub = res[res['factor'] == factor]
    if sub.empty:
        print(factor, 'is not a factor of', target, '| run', label); return
    row = sub.iloc[0]
    print(f"{target}  x  {factor}   (run={label})")
    print(f"  beta={row['beta']:+.4g}")
    betas = sb.load_betas(cfg, target, label, **read)
    if betas is not None and factor in betas.columns:
        b = betas[factor].to_numpy(dtype=float); b = b[~np.isnan(b)]; s = np.sort(b)
        mt = _min_trim_to_exclude(b)
        print(f"  n_sub={b.size}  zero={(b==0).sum()} pos={(b>0).sum()} neg={(b<0).sum()}  min_trim={mt}")
        for t in (0, 1, 2):
            if b.size >= 2 * t + 1:
                lo, hi = s[t], s[-(t + 1)]
                print(f"    drop {t}/side -> [{lo:+.3g}, {hi:+.3g}]  "
                      f"{'excludes 0' if (lo > 0 or hi < 0) else 'crosses 0'}")
        print(f"  => stars={default_stars({'min_trim': mt})}")
    else:
        lo = row.get('min_subsample_beta'); hi = row.get('max_subsample_beta')
        mt = 0.0 if (pd.notna(lo) and pd.notna(hi) and (lo > 0 or hi < 0)) else np.nan
        print("  (no raw-beta sidecar; only the 3-star 'no crossing' level is knowable from res)")
        print(f"  => stars={default_stars({'min_trim': mt})}")
    return row


def interaction_heatmap(inter, value='beta', *, stars=default_stars, min_stars=1,
                        only_significant_factors=True, factor_filter=is_slc_factor,
                        log=False, figsize=None, cmap='RdBu_r', title='',
                        star_legend='3=no crossing, 2=drop1/side, 1=drop2/side'):
    """Heatmap of (target x SLC factor) pairings colored by `value` (default point beta).
    `stars`(row->0..3) rates each pairing; `only_significant_factors` keeps only x-factors
    reaching `min_stars` somewhere (all of them -- no cap). `log` = sign(v)*log1p(|v|). NaN
    pairings render grey. Returns (value_matrix, stars_matrix)."""
    df = inter[inter['factor'].map(factor_filter)].copy()
    if df.empty:
        print('no factors pass factor_filter; nothing to plot'); return
    df['_stars'] = df.apply(stars, axis=1).astype(int)

    val = df.pivot_table(index='target', columns='factor', values=value, aggfunc='first')
    st = (df.pivot_table(index='target', columns='factor', values='_stars', aggfunc='first')
            .reindex(index=val.index, columns=val.columns).fillna(0).astype(int))
    if only_significant_factors:
        keep = (st >= min_stars).any(axis=0)
        val, st = val.loc[:, keep], st.loc[:, keep]
    if val.shape[1] == 0:
        print('no factor reaches min_stars=%d; loosen it or only_significant_factors=False'
              % min_stars); return

    M = val.to_numpy(dtype=float)
    if log:
        M = np.sign(M) * np.log1p(np.abs(M))
    vmax = float(np.nanmax(np.abs(M))) if np.isfinite(M).any() else 1.0
    cmap_obj = plt.get_cmap(cmap).copy(); cmap_obj.set_bad('lightgrey')
    figsize = figsize or (max(9, 1.2 * val.shape[1]), max(8, 0.45 * val.shape[0]))
    with plt.style.context('default'):   # isolate from the global seaborn theme
        fig, ax = plt.subplots(figsize=figsize)
        im = ax.imshow(np.ma.masked_invalid(M), aspect='auto', cmap=cmap_obj, vmin=-vmax, vmax=vmax)
        ax.set_xticks(range(val.shape[1])); ax.set_xticklabels(val.columns, rotation=45, ha='right', fontsize=7)
        ax.set_yticks(range(val.shape[0])); ax.set_yticklabels(val.index, fontsize=8)
        ax.set_xlabel('SLC transporter factor'); ax.set_ylabel('T-cell target signature')
        ax.set_title(f"{title + '  |  ' if title else ''}{value}"
                     f"{' (signed log1p)' if log else ''}   "
                     f"(stars: {star_legend})")
        S = st.to_numpy()
        for i in range(S.shape[0]):
            for j in range(S.shape[1]):
                if S[i, j] > 0:
                    ax.text(j, i, '*' * int(S[i, j]), ha='center', va='center',
                            fontsize=8, color='black')
        fig.colorbar(im, ax=ax, shrink=0.7, label=value)
        plt.tight_layout(); plt.show()
    return val, st

In [ ]:
# beta coefficients; SLC factors whose coefficient stays one-signed across subsamples.
if len(inter):
    interaction_heatmap(inter, title=RUN_LABEL)
else:
    print('no cached interactions -- run signature_run.py --config breast --factor-group slc')

# Understand any cell:  inspect_pair('HALLMARK_GLYCOLYSIS', slc_genes[0])
# Variations:
#   interaction_heatmap(inter, min_stars=3)                    # only 'no crossing' pairings
#   interaction_heatmap(inter, value='avg_subsample_beta')     # plot a subsample statistic
#   interaction_heatmap(inter, log=True)                       # signed-log scaling
#   interaction_heatmap(inter, only_significant_factors=False) # every SLC factor